# MP3 - SmallBank (B)
**Corey Farrow** · Comm3220 DM4DM

Server `f-sg6m-s4.comm.virginia.edu`, database `SmallBankDB` (T-SQL).

Join path: `LOAN_OFFICER` -> `LOAN` (lo_id) -> `CUSTOMER_IN_LOAN` (l_id) -> `CUSTOMER` (c_id). `INSURANCE_PLAN` hangs off `CUSTOMER` (c_id) and is optional, so it is reached with a LEFT JOIN.

Formatting rules: money `FORMAT(x,'C0')`, integers `'N0'`, interest rates `'N3'`. `FORMAT` returns text, so sorting always uses the unformatted value.

**Q1:** List our customers who have a loan, where they live, their loan officers, and their phones. Alphabetical by city and then by customer last and first name. Column names specify who is who.

In [ ]:
SELECT DISTINCT
    c.c_id     AS 'Customer ID',
    c.l_name   AS 'Customer Last Name',
    c.f_name   AS 'Customer First Name',
    c.city     AS 'Customer City',
    c.state    AS 'Customer State',
    lo.lo_id   AS 'Loan Officer ID',
    lo.l_name  AS 'Loan Officer Last Name',
    lo.f_name  AS 'Loan Officer First Name',
    lo.phone   AS 'Loan Officer Phone'
FROM CUSTOMER c
    JOIN CUSTOMER_IN_LOAN cil ON c.c_id = cil.c_id
    JOIN LOAN l               ON cil.l_id = l.l_id
    JOIN LOAN_OFFICER lo      ON l.lo_id = lo.lo_id
ORDER BY c.city, c.l_name, c.f_name;

**Q2:** Who are Carl Dyer's (id = 8) clients? Alphabetized list by last and first name.

In [ ]:
-- DISTINCT: a client with two of Carl's loans should appear once.
SELECT DISTINCT
    c.c_id    AS 'Customer ID',
    c.l_name  AS 'Last Name',
    c.f_name  AS 'First Name'
FROM CUSTOMER c
    JOIN CUSTOMER_IN_LOAN cil ON c.c_id = cil.c_id
    JOIN LOAN l               ON cil.l_id = l.l_id
WHERE l.lo_id = 8
ORDER BY c.l_name, c.f_name;

**Q3:** We measure loan officer performance in terms of the amount they manage. Who manages the least money that is not yet past due as of the time the query is run?

In [ ]:
-- WITH TIES returns every officer tied for the lowest total.
SELECT TOP 1 WITH TIES
    lo.lo_id                   AS 'Loan Officer ID',
    lo.l_name                  AS 'Last Name',
    lo.f_name                  AS 'First Name',
    FORMAT(SUM(l.principal), 'C0') AS 'Amount Managed (Not Past Due)'
FROM LOAN_OFFICER lo
    JOIN LOAN l ON lo.lo_id = l.lo_id
WHERE l.date_due >= CAST(GETDATE() AS DATE)
GROUP BY lo.lo_id, lo.l_name, lo.f_name
ORDER BY SUM(l.principal) ASC;

**Q4:** List all customers alphabetically, including insurance information for those who have it. Print out 0 (zero) or '-' rather than 'Null' if they do not.

In [ ]:
-- LEFT JOIN keeps customers with no plan; ISNULL swaps the NULLs for '-'.
SELECT
    c.c_id     AS 'Customer ID',
    c.l_name   AS 'Last Name',
    c.f_name   AS 'First Name',
    ISNULL(FORMAT(ip.coverage, 'C0'), '-') AS 'Insurance Coverage',
    ISNULL(FORMAT(ip.premium,  'C0'), '-') AS 'Insurance Premium'
FROM CUSTOMER c
    LEFT JOIN INSURANCE_PLAN ip ON c.c_id = ip.c_id
ORDER BY c.l_name, c.f_name;

**Q5:** Create a report showing which loan officers has offered an average rate less than 13%. Order by rate ascending.

In [ ]:
-- The condition is on an aggregate, so it goes in HAVING. Rates are percent numbers (13 = 13%).
SELECT
    lo.lo_id                 AS 'Loan Officer ID',
    lo.l_name                AS 'Last Name',
    lo.f_name                AS 'First Name',
    FORMAT(AVG(l.rate), 'N3') AS 'Average Rate Offered (%)'
FROM LOAN_OFFICER lo
    JOIN LOAN l ON lo.lo_id = l.lo_id
GROUP BY lo.lo_id, lo.l_name, lo.f_name
HAVING AVG(l.rate) < 13
ORDER BY AVG(l.rate) ASC;

**Q6:** Who are our top quartile borrowers? Show customers and total borrowed. Order by total principal borrowed, largest on top.

In [ ]:
SELECT TOP 25 PERCENT
    c.c_id     AS 'Customer ID',
    c.l_name   AS 'Last Name',
    c.f_name   AS 'First Name',
    FORMAT(SUM(l.principal), 'C0') AS 'Total Borrowed'
FROM CUSTOMER c
    JOIN CUSTOMER_IN_LOAN cil ON c.c_id = cil.c_id
    JOIN LOAN l               ON cil.l_id = l.l_id
GROUP BY c.c_id, c.l_name, c.f_name
ORDER BY SUM(l.principal) DESC;

**Q7:** Who borrowed the most money in Richmond? (do not show the $ amount).

In [ ]:
-- Sort on the total but do not display it, as instructed.
SELECT TOP 1 WITH TIES
    c.c_id    AS 'Customer ID',
    c.l_name  AS 'Last Name',
    c.f_name  AS 'First Name',
    c.city    AS 'City',
    c.state   AS 'State'
FROM CUSTOMER c
    JOIN CUSTOMER_IN_LOAN cil ON c.c_id = cil.c_id
    JOIN LOAN l               ON cil.l_id = l.l_id
WHERE c.city = 'Richmond'
GROUP BY c.c_id, c.l_name, c.f_name, c.city, c.state
ORDER BY SUM(l.principal) DESC;

**Q8:** How many clients does Carl Dyer have? Assume you know his lo_id.

In [ ]:
-- COUNT(DISTINCT) because one client can hold several of Carl's loans.
SELECT FORMAT(COUNT(DISTINCT cil.c_id), 'N0') AS 'Number of Carl Dyer''s Clients'
FROM LOAN l
    JOIN CUSTOMER_IN_LOAN cil ON l.l_id = cil.l_id
WHERE l.lo_id = 8;

**Q9:** What is our insurance exposure by city? (Exposure = sum of coverages)

In [ ]:
SELECT
    c.city   AS 'City',
    c.state  AS 'State',
    FORMAT(SUM(ip.coverage), 'C0') AS 'Insurance Exposure'
FROM CUSTOMER c
    JOIN INSURANCE_PLAN ip ON c.c_id = ip.c_id
GROUP BY c.city, c.state
ORDER BY SUM(ip.coverage) DESC;

**Q10:** How many of our Charlottesville customers are not insured?

In [ ]:
-- Anti-join: LEFT JOIN, then keep the rows with no matching plan.
SELECT FORMAT(COUNT(*), 'N0') AS 'Uninsured Charlottesville Customers'
FROM CUSTOMER c
    LEFT JOIN INSURANCE_PLAN ip ON c.c_id = ip.c_id
WHERE c.city = 'Charlottesville'
  AND ip.c_id IS NULL;

**Q11:** List all the loan officers and how many insured customers each has. Alphabetic by officer.

In [ ]:
-- LEFT JOINs all the way down so officers with no loans or no insured clients still show 0.
-- COUNT(DISTINCT ip.c_id) counts only customers who have a plan, each once.
SELECT
    lo.lo_id   AS 'Loan Officer ID',
    lo.l_name  AS 'Last Name',
    lo.f_name  AS 'First Name',
    FORMAT(COUNT(DISTINCT ip.c_id), 'N0') AS 'Insured Customers'
FROM LOAN_OFFICER lo
    LEFT JOIN LOAN l               ON lo.lo_id = l.lo_id
    LEFT JOIN CUSTOMER_IN_LOAN cil ON l.l_id = cil.l_id
    LEFT JOIN INSURANCE_PLAN ip    ON cil.c_id = ip.c_id
GROUP BY lo.lo_id, lo.l_name, lo.f_name
ORDER BY lo.l_name, lo.f_name;

**Q12:** List all the loan officers who have customers in Charlottesville, order by total amount managed in that town, largest on top.

In [ ]:
-- DISTINCT subquery so a loan shared by two Charlottesville co-borrowers is counted once.
SELECT
    lo.lo_id   AS 'Loan Officer ID',
    lo.l_name  AS 'Last Name',
    lo.f_name  AS 'First Name',
    FORMAT(SUM(cvl.principal), 'C0') AS 'Amount Managed in Charlottesville'
FROM LOAN_OFFICER lo
    JOIN (SELECT DISTINCT l.l_id, l.lo_id, l.principal
          FROM LOAN l
              JOIN CUSTOMER_IN_LOAN cil ON l.l_id = cil.l_id
              JOIN CUSTOMER c           ON cil.c_id = c.c_id
          WHERE c.city = 'Charlottesville') cvl
        ON lo.lo_id = cvl.lo_id
GROUP BY lo.lo_id, lo.l_name, lo.f_name
ORDER BY SUM(cvl.principal) DESC;